# 트랜스포머를 위한 Python & NumPy 기초 - 행렬 연산 기초

- Tutorial ID: `tut-0-1`
- Tutorial: 트랜스포머를 위한 Python & NumPy 기초
- Section ID: `s0-1-1`
- Section: 행렬 연산 기초


In [ ]:
# ============================================================
# 코드 읽는 법 — 행렬 연산 기초
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

In [ ]:
# 1. 기본 행렬 연산

In [ ]:
print("=" * 50)
print("1. 기본 행렬 연산")
print("=" * 50)

# 임베딩 차원 설정 (실제 GPT-2 small: d_model=768)
d_model = 4  # 작은 예시

# 랜덤 가중치 행렬 초기화 (Xavier 초기화)
np.random.seed(42)
W = np.random.randn(d_model, d_model) / np.sqrt(d_model)
x = np.random.randn(d_model)  # 단일 토큰 임베딩

print(f"가중치 행렬 W (shape: {W.shape}):")
print(np.round(W, 3))
print(f"
입력 벡터 x (shape: {x.shape}):")
print(np.round(x, 3))

# 행렬-벡터 곱
y = W @ x  # 또는 np.matmul(W, x)
print(f"
W @ x = y (shape: {y.shape}):")
print(np.round(y, 3))

In [ ]:
# 2. 소프트맥스 - 어텐션 패턴의 핵심

In [ ]:
print("
" + "=" * 50)
print("2. 소프트맥스 (Numerically Stable)")
print("=" * 50)

def softmax(x, axis=-1):
    """수치적으로 안정적인 소프트맥스"""
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)  # 오버플로우 방지
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

# 어텐션 점수 시뮬레이션
attn_scores = np.array([2.0, 1.0, 0.5, -1.0])  # 4개의 토큰
attn_probs = softmax(attn_scores)
print(f"어텐션 점수: {attn_scores}")
print(f"소프트맥스 결과: {np.round(attn_probs, 4)}")
print(f"합계 = {attn_probs.sum():.6f} (항상 1)")

In [ ]:
# 3. 잔차 스트림의 선형성

In [ ]:
print("
" + "=" * 50)
print("3. 잔차 스트림 (Residual Stream)")
print("=" * 50)

# 트랜스포머의 잔차 연결: x = x + attention_output + mlp_output
x_0 = np.array([1.0, 0.5, -0.3, 0.8])  # 초기 임베딩
attn_out = np.array([0.1, -0.2, 0.4, 0.0])  # 어텐션 헤드 출력
mlp_out = np.array([-0.05, 0.3, 0.1, -0.2])  # MLP 출력

x_1 = x_0 + attn_out  # 어텐션 후 잔차 스트림
x_2 = x_1 + mlp_out   # MLP 후 잔차 스트림

print(f"초기 임베딩 x_0: {x_0}")
print(f"어텐션 출력: {attn_out}")
print(f"x_1 = x_0 + attn: {x_1}")
print(f"MLP 출력: {mlp_out}")
print(f"x_2 = x_1 + mlp: {x_2}")
print()
print("핵심: 잔차 스트림은 순수하게 선형적(additive)!")
print("어떤 계층도 이전 정보를 '지우지 않고' 더합니다.")

In [ ]:
# 4. 가상 가중치 (Virtual Weights)

In [ ]:
print("
" + "=" * 50)
print("4. 가상 가중치 (Virtual Weights)")
print("=" * 50)

# W_E: 임베딩 행렬 (vocab_size -> d_model)
# W_U: 언임베딩 행렬 (d_model -> vocab_size)
# 가상 가중치: W_U @ W_E (직접 연결!)

vocab_size = 8
d_model = 4

W_E = np.random.randn(vocab_size, d_model) / np.sqrt(d_model)  # 임베딩
W_U = np.random.randn(d_model, vocab_size) / np.sqrt(vocab_size)  # 언임베딩

# 가상 가중치: 임베딩과 언임베딩을 직접 연결
W_virtual = W_U.T @ W_E.T  # (vocab x vocab) - 바이그램 통계!
print(f"임베딩 행렬 W_E: {W_E.shape}")
print(f"언임베딩 행렬 W_U: {W_U.shape}")
print(f"가상 가중치 W_U·W_E: {W_virtual.shape}")
print("이 행렬은 '0-레이어 트랜스포머'의 바이그램 테이블입니다!")